# Notebook 03c v2 — NSL Dirichlet calibration (parallel comparison)

**What this notebook does:**

Fits Dirichlet calibration (Kull et al. NeurIPS 2019, "Beyond temperature scaling") on the calibration set carved out in Notebook 01 v2, applies it to the test set, and reports per-class Brier and ECE. Outputs are saved alongside the hybrid Platt/isotonic results from Notebook 03 v2 for direct comparison.

**Why Dirichlet calibration:**

Per-class one-vs-rest isotonic with row-normalisation has two limitations on distribution-shifted test sets like NSL-KDD:
1. Calibrator fit on calib distribution doesn't transfer when test distribution differs
2. Row-normalisation step can amplify per-class errors

Dirichlet calibration is a proper multiclass extension of Platt scaling. It learns a joint calibrator over all classes simultaneously (no per-class fits, no renormalisation step). It cannot fix distribution shift (no calibrator can), but it removes the renormalisation source of error.

**Variant used: ODIR** (Off-Diagonal and Intercept Regularisation, the regularised variant). ODIR is more stable than full-Dirichlet on rare classes like NSL U2R (n_calib=10).

**Comparison story (paper-ready):**

Three calibration approaches benchmarked:
- Pre-calibration (baseline)
- Hybrid Platt/isotonic (Notebook 03 v2 — addresses TA's rare-class sample-size concern)
- Dirichlet ODIR (this notebook — SOTA multiclass approach, no renormalisation)

**Implementation:** uses the `dirichletcal` package, Kull et al's official reference implementation.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
# Install Dirichlet calibration package (Kull et al. 2019 reference implementation)
!pip install -q dirichletcal

# Verify install
from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator
print('✓ dirichletcal package loaded')

  Preparing metadata (setup.py) ... done
✓ dirichletcal package loaded


In [3]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter

from sklearn.metrics import brier_score_loss
from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASET = 'nsl_kdd_v2'
PROCESSED = Path(REPO) / 'data' / 'processed' / DATASET
MODELS_DIR = Path(REPO) / 'models' / DATASET
PREDS_DIR = MODELS_DIR / 'predictions'
CALIB_DIR_HYBRID = Path(REPO) / 'calibrators' / DATASET
CALIB_DIR_DIRICHLET = Path(REPO) / 'calibrators' / f'{DATASET}_dirichlet'
CALIB_DIR_DIRICHLET.mkdir(parents=True, exist_ok=True)

ECE_N_BINS = 10

print(f'Dataset: {DATASET}')
print(f'Hybrid outputs dir: {CALIB_DIR_HYBRID}')
print(f'Dirichlet outputs dir: {CALIB_DIR_DIRICHLET}')

Dataset: nsl_kdd_v2
Hybrid outputs dir: /content/drive/MyDrive/XIDS_Research/xids-research/calibrators/nsl_kdd_v2
Dirichlet outputs dir: /content/drive/MyDrive/XIDS_Research/xids-research/calibrators/nsl_kdd_v2_dirichlet


## 2. Load labels and class mappings

In [4]:
y_calib_b = np.load(PROCESSED / 'y_calib_binary.npy')
y_calib_5 = np.load(PROCESSED / 'y_calib_5class.npy')
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'Calibration set: {len(y_calib_b):,}, Test set: {len(y_test_b):,}')
print()
print('Per-class calibration sizes:')
calib_counts_5 = Counter(y_calib_5)
for c in range(5):
    print(f'  {CLASS_NAMES_5[c]:8s}: n={calib_counts_5[c]:>6,}')

Calibration set: 25,195, Test set: 22,544

Per-class calibration sizes:
  Normal  : n=13,469
  DoS     : n= 9,186
  Probe   : n= 2,331
  R2L     : n=   199
  U2R     : n=    10


## 3. Helper functions

In [5]:
def expected_calibration_error(probs, labels, n_bins=10):
    """ECE computed on one-vs-rest binary probabilities."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (probs >= lo) & (probs <= hi)
        else:
            mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        bin_conf = probs[mask].mean()
        bin_acc = labels[mask].mean()
        weight = mask.sum() / n
        ece += weight * abs(bin_conf - bin_acc)
    return float(ece)

def safe_log_probs(p, eps=1e-12):
    """Dirichlet calibration works on log-probabilities. Clip to avoid log(0)."""
    return np.log(np.clip(p, eps, 1.0))

def fit_dirichlet_5class(p_calib_2d, y_calib, reg_lambda=1e-3):
    """Fit ODIR (regularised Dirichlet) calibrator on 5-class probabilities.

    Args:
        p_calib_2d: calib probs, shape (n_calib, 5)
        y_calib: integer labels, shape (n_calib,)
        reg_lambda: regularisation strength (small for ODIR)

    Returns:
        fitted FullDirichletCalibrator
    """
    cal = FullDirichletCalibrator(reg_lambda=reg_lambda, reg_mu=None)
    cal.fit(p_calib_2d, y_calib)
    return cal

def evaluate_calibrator(p_test_cal, y_test, n_classes):
    """Per-class Brier and ECE on calibrated probabilities."""
    brier = {}
    ece = {}
    for c in range(n_classes):
        y_c = (y_test == c).astype(int)
        p_c = p_test_cal[:, c]
        brier[c] = float(brier_score_loss(y_c, p_c))
        ece[c] = expected_calibration_error(p_c, y_c, ECE_N_BINS)
    return brier, ece

print('✓ Helper functions loaded')

✓ Helper functions loaded


## 4. Calibrate all 9 models with Dirichlet

In [6]:
ALL_MODELS = [
    ('rf_binary_cw',     'binary'),
    ('xgb_binary_cw',    'binary'),
    ('dnn_binary_cw',    'binary'),
    ('rf_5class_smote',  '5class'),
    ('xgb_5class_smote', '5class'),
    ('dnn_5class_smote', '5class'),
    ('rf_5class_cw',     '5class'),
    ('xgb_5class_cw',    '5class'),
    ('dnn_5class_cw',    '5class'),
]

calibration_summary = {}

for name, task in ALL_MODELS:
    print(f'\nCalibrating {name} ({task}) with Dirichlet ODIR...')

    p_calib = np.load(PREDS_DIR / f'{name}_calib_proba.npy')
    p_test = np.load(PREDS_DIR / f'{name}_test_proba.npy')

    # Ensure 2D shape
    if p_calib.ndim == 1:
        p_calib = np.column_stack([1 - p_calib, p_calib])
    if p_test.ndim == 1:
        p_test = np.column_stack([1 - p_test, p_test])

    n_classes = p_calib.shape[1]
    y_calib_use = y_calib_b if task == 'binary' else y_calib_5
    y_test_use = y_test_b if task == 'binary' else y_test_5

    # Pre-cal metrics for comparison
    brier_pre, ece_pre = evaluate_calibrator(p_test, y_test_use, n_classes)

    # Fit Dirichlet calibrator
    try:
        cal = fit_dirichlet_5class(p_calib, y_calib_use)
        p_test_cal = cal.predict_proba(p_test)
        success = True
    except Exception as e:
        print(f'  ✗ Dirichlet fit failed: {e}')
        print(f'    Falling back to identity (no calibration) for this model.')
        p_test_cal = p_test.copy()
        success = False

    # Post-cal metrics
    brier_post, ece_post = evaluate_calibrator(p_test_cal, y_test_use, n_classes)

    # Save calibrated probabilities
    np.save(CALIB_DIR_DIRICHLET / f'{name}_test_proba_calibrated.npy', p_test_cal)

    # Report
    if task == '5class':
        for c in range(5):
            cname = CLASS_NAMES_5[c]
            print(f'  {cname:8s}: Brier {brier_pre[c]:.4f} → {brier_post[c]:.4f}'
                  f'   ECE {ece_pre[c]:.4f} → {ece_post[c]:.4f}')
    else:
        print(f'  Attack  : Brier {brier_pre[1]:.4f} → {brier_post[1]:.4f}'
              f'   ECE {ece_pre[1]:.4f} → {ece_post[1]:.4f}')

    calibration_summary[name] = {
        'task': task,
        'method': 'dirichlet_odir' if success else 'failed_identity',
        'success': success,
        'brier_pre':  {int(k): v for k, v in brier_pre.items()},
        'brier_post': {int(k): v for k, v in brier_post.items()},
        'ece_pre':  {int(k): v for k, v in ece_pre.items()},
        'ece_post': {int(k): v for k, v in ece_post.items()},
    }

with open(CALIB_DIR_DIRICHLET / 'calibration_summary.json', 'w') as f:
    json.dump(calibration_summary, f, indent=2)

print(f'\n✓ All 9 models calibrated with Dirichlet.')
print(f'  Summary: {CALIB_DIR_DIRICHLET}/calibration_summary.json')


Calibrating rf_binary_cw (binary) with Dirichlet ODIR...
  Attack  : Brier 0.1633 → 0.1663   ECE 0.1900 → 0.1919

Calibrating xgb_binary_cw (binary) with Dirichlet ODIR...
  Attack  : Brier 0.1988 → 0.1929   ECE 0.2078 → 0.2054

Calibrating dnn_binary_cw (binary) with Dirichlet ODIR...
  Attack  : Brier 0.1789 → 0.1772   ECE 0.1803 → 0.1789

Calibrating rf_5class_smote (5class) with Dirichlet ODIR...
  Normal  : Brier 0.1509 → 0.1726   ECE 0.1810 → 0.2009
  DoS     : Brier 0.0679 → 0.0702   ECE 0.0674 → 0.0723
  Probe   : Brier 0.0361 → 0.0388   ECE 0.0227 → 0.0257
  R2L     : Brier 0.1045 → 0.1141   ECE 0.1089 → 0.1181
  U2R     : Brier 0.0021 → 0.0029   ECE 0.0028 → 0.0050

Calibrating xgb_5class_smote (5class) with Dirichlet ODIR...
  Normal  : Brier 0.1824 → 0.1833   ECE 0.1943 → 0.1979
  DoS     : Brier 0.0629 → 0.0637   ECE 0.0652 → 0.0660
  Probe   : Brier 0.0412 → 0.0407   ECE 0.0376 → 0.0368
  R2L     : Brier 0.1089 → 0.1123   ECE 0.1119 → 0.1157
  U2R     : Brier 0.0023 → 0.

## 5. Three-way comparison: pre-cal vs hybrid vs Dirichlet

In [7]:
# Load hybrid (Platt/isotonic) summary from Notebook 03 v2
with open(CALIB_DIR_HYBRID / 'calibration_summary.json') as f:
    hybrid_summary = json.load(f)

# Build comparison table: per-model macro Brier and ECE for each method
rows = []
for name, task in ALL_MODELS:
    d_pre = calibration_summary[name]['brier_pre']
    d_post = calibration_summary[name]['brier_post']
    h = hybrid_summary[name]
    h_pre = h['brier_pre']
    h_post = h['brier_post']

    # Macro Brier (mean over classes that have data)
    if task == '5class':
        b_pre = np.mean([d_pre[str(c)] for c in range(5)]) if isinstance(list(d_pre.keys())[0], str) else np.mean([d_pre[c] for c in range(5)])
        b_hybrid = np.mean([h_post[str(c)] for c in range(5)])
        b_dirichlet = np.mean([d_post[c] for c in range(5)])
    else:
        b_pre = d_pre[1]
        b_hybrid = h_post['1']
        b_dirichlet = d_post[1]

    # Identify winner
    methods = {'pre': b_pre, 'hybrid': b_hybrid, 'dirichlet': b_dirichlet}
    winner = min(methods, key=methods.get)

    rows.append({
        'Model': name,
        'Task': '5-class' if task == '5class' else 'binary',
        'Brier pre':       round(b_pre,       5),
        'Brier hybrid':    round(b_hybrid,    5),
        'Brier dirichlet': round(b_dirichlet, 5),
        'Winner': winner,
    })

df_cmp = pd.DataFrame(rows)
print('NSL v2 — Three-way calibration comparison (macro Brier)')
print('=' * 100)
print(df_cmp.to_string(index=False))
print('=' * 100)

TABLES_DIR = Path(REPO) / 'results' / 'tables'
csv_path = TABLES_DIR / 'nslkdd_v2_calibration_threeway.csv'
df_cmp.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

# Count winners
winners = df_cmp['Winner'].value_counts()
print(f'\nWinner counts: {winners.to_dict()}')

NSL v2 — Three-way calibration comparison (macro Brier)
           Model    Task  Brier pre  Brier hybrid  Brier dirichlet    Winner
    rf_binary_cw  binary    0.16328       0.16747          0.16628       pre
   xgb_binary_cw  binary    0.19880       0.19408          0.19292 dirichlet
   dnn_binary_cw  binary    0.17887       0.18054          0.17723 dirichlet
 rf_5class_smote 5-class    0.07231       0.08323          0.07973       pre
xgb_5class_smote 5-class    0.07953       0.08039          0.08053       pre
dnn_5class_smote 5-class    0.08437       0.08734          0.08626       pre
    rf_5class_cw 5-class    0.07914       0.08641          0.08375       pre
   xgb_5class_cw 5-class    0.08033       0.08012          0.08255    hybrid
   dnn_5class_cw 5-class    0.08041       0.08580          0.08352       pre

Saved: /content/drive/MyDrive/XIDS_Research/xids-research/results/tables/nslkdd_v2_calibration_threeway.csv

Winner counts: {'pre': 6, 'dirichlet': 2, 'hybrid': 1}


## 6. Per-class three-way comparison (5-class models only)

In [8]:
perclass_rows = []
for name, task in ALL_MODELS:
    if task != '5class':
        continue
    d = calibration_summary[name]
    h = hybrid_summary[name]

    for c in range(5):
        d_pre = d['brier_pre'][c]
        d_post = d['brier_post'][c]
        h_post = h['brier_post'][str(c)]
        h_strategy = h['strategies'][str(c)]

        methods = {'pre': d_pre, 'hybrid': h_post, 'dirichlet': d_post}
        winner = min(methods, key=methods.get)

        perclass_rows.append({
            'Model': name,
            'Class': CLASS_NAMES_5[c],
            'n_calib': calib_counts_5[c],
            'Brier pre': round(d_pre, 5),
            'Brier hybrid': round(h_post, 5),
            'Hybrid strategy': h_strategy,
            'Brier dirichlet': round(d_post, 5),
            'Winner': winner,
        })

df_perclass = pd.DataFrame(perclass_rows)
print('NSL v2 — Per-class three-way comparison')
print('=' * 130)
print(df_perclass.to_string(index=False))
print('=' * 130)

csv_path = TABLES_DIR / 'nslkdd_v2_calibration_threeway_perclass.csv'
df_perclass.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

# Aggregate: which method wins for which class?
print(f'\nWinners by class:')
for c in range(5):
    cname = CLASS_NAMES_5[c]
    cnt = Counter(df_perclass[df_perclass['Class'] == cname]['Winner'])
    print(f'  {cname:8s}: {dict(cnt)}')

NSL v2 — Per-class three-way comparison
           Model  Class  n_calib  Brier pre  Brier hybrid Hybrid strategy  Brier dirichlet    Winner
 rf_5class_smote Normal    13469    0.15093       0.19154        isotonic          0.17264       pre
 rf_5class_smote    DoS     9186    0.06785       0.06095        isotonic          0.07023    hybrid
 rf_5class_smote  Probe     2331    0.03612       0.04148        isotonic          0.03879       pre
 rf_5class_smote    R2L      199    0.10448       0.11932        isotonic          0.11413       pre
 rf_5class_smote    U2R       10    0.00214       0.00289           platt          0.00287       pre
xgb_5class_smote Normal    13469    0.18238       0.18405        isotonic          0.18326       pre
xgb_5class_smote    DoS     9186    0.06288       0.06199        isotonic          0.06372    hybrid
xgb_5class_smote  Probe     2331    0.04124       0.03969        isotonic          0.04075    hybrid
xgb_5class_smote    R2L      199    0.10892       0

## 7. Sanity checks

In [9]:
print('Sanity checks on Dirichlet calibrated probabilities:')
print('-' * 80)

for name, task in ALL_MODELS:
    p_cal = np.load(CALIB_DIR_DIRICHLET / f'{name}_test_proba_calibrated.npy')
    expected_cols = 2 if task == 'binary' else 5

    shape_ok = p_cal.shape[1] == expected_cols
    range_ok = (p_cal >= 0).all() and (p_cal <= 1).all()
    row_sums = p_cal.sum(axis=1)
    sum_ok = np.allclose(row_sums, 1, atol=0.001)
    max_dev = float(np.abs(row_sums - 1).max())
    finite_ok = np.isfinite(p_cal).all()

    all_ok = shape_ok and range_ok and sum_ok and finite_ok
    flag = '✓' if all_ok else '✗'
    print(f'  {flag} {name:<22} shape={p_cal.shape}  range_ok={range_ok}  '
          f'sum_max_dev={max_dev:.5f}  finite={finite_ok}')

# Key advantage of Dirichlet over hybrid: should sum to 1 by construction (no renorm)
print()
print('Note: Dirichlet calibrated probs should sum to 1 by construction')
print('(unlike per-class isotonic which needed explicit renormalisation).')

Sanity checks on Dirichlet calibrated probabilities:
--------------------------------------------------------------------------------
  ✓ rf_binary_cw           shape=(22544, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_binary_cw          shape=(22544, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_binary_cw          shape=(22544, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ rf_5class_smote        shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_5class_smote       shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_5class_smote       shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ rf_5class_cw           shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_5class_cw          shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_5class_cw          shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True

Note: Dirichlet calibrated p

## 8. Commit

In [10]:
os.chdir(REPO)
!git add notebooks/03c_nsl_calibration_dirichlet_v2.ipynb
!git add calibrators/nsl_kdd_v2_dirichlet/
!git add results/tables/nslkdd_v2_calibration_threeway.csv
!git add results/tables/nslkdd_v2_calibration_threeway_perclass.csv
!git status --short
!git commit -m 'Notebook 03c-NSL v2: Dirichlet calibration parallel comparison (pre-cal vs hybrid vs Dirichlet)'
!git push origin main

Refresh index: 100% (243/243), done.
A  calibrators/nsl_kdd_v2_dirichlet/calibration_summary.json
 M notebooks/01_cic_data_exploration_v2.ipynb
 M notebooks/01_data_exploration_v2.ipynb
 M notebooks/01_unsw_data_exploration_v2.ipynb
 M notebooks/02_cic_train_models_v2.ipynb
 M notebooks/02_nsl_rf_retrain_v2.ipynb
 M notebooks/02_train_models_v2.ipynb
 M notebooks/02_unsw_train_models_v2.ipynb
 M notebooks/03_nsl_calibration_v2.ipynb
AM notebooks/03c_nsl_calibration_dirichlet_v2.ipynb
A  results/tables/nslkdd_v2_calibration_threeway.csv
A  results/tables/nslkdd_v2_calibration_threeway_perclass.csv
?? calibrators/nsl_kdd/
?? calibrators/unsw_nb15/
?? docs/v2_rebuild_progress_day1_day2_v7.md
?? models/
?? notebooks/02b_unsw_dnn_diagnostic.ipynb
?? notebooks/03b_nsl_calibration_diagnostic.ipynb
?? results/figures/diag_nsl_dnn5smote_normal_reliability.png
?? results/figures/diag_nsl_rf5smote_normal_reliability.png
?? results/figures/unsw_dnn_5class_diagnostic.png
?? shap_values/unsw_nb15/
[